# Plantilla — Ciclo Estadístico Completo

> **Uso:** copiar esta plantilla para cada nueva línea temática.  
> Ajustar: `LINEA`, `VARIABLE`, `UNIDAD`, `DATE_COL`, `ruta del archivo`.  
> Consultar `Plan.md` sección 3 y `docs/fuentes/<linea>.md` antes de ejecutar.

```
Flujo: Carga → Validación → EDA → Descriptiva → Inferencial
       → Preprocesamiento → Modelos → Backtesting → Reporte
```

## 0. Configuración del proyecto

In [ ]:
# ── Ajustar por línea temática ──────────────────────────────────────
LINEA    = "calidad_aire"          # key en docs/fuentes/
VARIABLE = "pm25"                  # columna objetivo
UNIDAD   = "µg/m³"
DATE_COL = "fecha"
DOMINIO  = "air_quality"           # general | hydrology | air_quality
HORIZONTE = 7                      # pasos de pronóstico
N_SPLITS  = 4                      # folds de backtesting
# ────────────────────────────────────────────────────────────────────

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import estadistica_ambiental as ea
from estadistica_ambiental.eda.viz import (
    plot_series, plot_missing_heatmap, plot_histogram,
    plot_seasonal_means, plot_correlation_heatmap,
)
from estadistica_ambiental.descriptive.temporal import (
    decompose_stl, acf_values, pacf_values,
)
from estadistica_ambiental.descriptive.bivariate import correlation_table
from estadistica_ambiental.inference.distributions import normality_tests
from estadistica_ambiental.inference.hypothesis import mannwhitney
from estadistica_ambiental.config import NORMA_CO, NORMA_OMS

print(f"estadistica_ambiental v{ea.__version__}")
print("Modelos disponibles:", ea.list_models())

## 1. Ingesta de datos

In [ ]:
# Opción A: cargar desde archivo real
# df = ea.load(f"data/raw/{LINEA}.csv", date_col=DATE_COL)

# Opción B: datos sintéticos de prueba (reemplazar con reales)
np.random.seed(42)
n = 120
df = pd.DataFrame({
    DATE_COL: pd.date_range("2015-01-01", periods=n, freq="ME"),
    VARIABLE: np.random.gamma(3, 5, n) + np.linspace(0, 8, n),
    "temperatura": np.random.normal(14, 3, n),
    "humedad":     np.random.normal(70, 10, n),
})
# Introducir 5% de faltantes
df.loc[df.sample(frac=0.05).index, VARIABLE] = np.nan

print(f"Shape: {df.shape} | Faltantes: {df[VARIABLE].isna().sum()}")
df.head()

## 2. Validación de calidad (física + estadística)

In [ ]:
# Rangos físicos y duplicados
val = ea.validate(df, date_col=DATE_COL)
print(val.summary())

In [ ]:
# Catálogo de variables
cat = ea.classify(df)
print(cat.summary())

In [ ]:
# Calidad profunda: gaps, freeze, outliers estadísticos
quality = ea.assess_quality(df, date_col=DATE_COL)
print(quality.summary())

## 3. EDA automático

In [ ]:
report_path = ea.run_eda(
    df,
    output=f"data/output/reports/eda_{LINEA}.html",
    title=f"EDA — {LINEA.replace('_', ' ').title()}",
    date_col=DATE_COL,
    use_ydata=False,
)
print(f"Reporte EDA: {report_path}")

## 4. Visualización exploratoria

In [ ]:
fig = plot_series(df, DATE_COL, VARIABLE,
                  title=f"{VARIABLE} ({UNIDAD})")
plt.show()

In [ ]:
plot_missing_heatmap(df, date_col=DATE_COL)
plt.show()

In [ ]:
plot_seasonal_means(df, DATE_COL, VARIABLE, period="month")
plt.show()

In [ ]:
plot_histogram(df, VARIABLE)
plt.show()

In [ ]:
plot_correlation_heatmap(df)
plt.show()

## 5. Estadística descriptiva

In [ ]:
ea.summarize(df)

In [ ]:
corr = correlation_table(df, method="spearman")
corr.head(10)

## 6. Estadística inferencial

In [ ]:
# Prueba de normalidad
normality_tests(df[VARIABLE].dropna())

In [ ]:
# Estacionariedad (obligatorio antes de ARIMA — ADR-004)
ts_index = df.set_index(DATE_COL)[VARIABLE].dropna()
ea.stationarity_report(ts_index)

In [ ]:
# Tendencia Mann-Kendall
mk = ea.mann_kendall(ts_index)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f} {UNIDAD}/período")

In [ ]:
# Descomposición STL
from estadistica_ambiental.descriptive.temporal import decompose_stl
stl = decompose_stl(ts_index, period=12)
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, col, lbl in zip(axes, ["observed","trend","seasonal","residual"],
                         ["Observado","Tendencia","Estacionalidad","Residuo"]):
    ax.plot(stl[col], color="#1a5276", lw=1)
    ax.set_ylabel(lbl, fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle(f"STL — {VARIABLE}", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ACF y PACF
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
acf = acf_values(ts_index, nlags=24)
pacf_v = pacf_values(ts_index, nlags=24)
ax1.stem(acf, markerfmt="C0o", linefmt="C0-", basefmt="k-"); ax1.set_title("ACF")
ax2.stem(pacf_v, markerfmt="C1o", linefmt="C1-", basefmt="k-"); ax2.set_title("PACF")
plt.tight_layout(); plt.show()

In [ ]:
# Probabilidad de excedencia (si aplica norma)
if VARIABLE in ("pm25", "pm10"):
    norma_key = f"{VARIABLE}_24h"
    if norma_key in NORMA_CO:
        exc = ea.exceedance_probability(df[VARIABLE].dropna(), NORMA_CO[norma_key])
        print(f"Excedencias norma CO ({NORMA_CO[norma_key]} {UNIDAD}): "
              f"{exc['n_exceedances']} días ({exc['pct_exceed']:.1f}%)")

## 7. Preprocesamiento

In [ ]:
# Imputación de faltantes
df_clean = ea.impute(df.copy(), cols=[VARIABLE], method="linear")
print(f"Faltantes antes={df[VARIABLE].isna().sum()} | después={df_clean[VARIABLE].isna().sum()}")

## 8. Modelos predictivos — comparación

In [ ]:
ts = df_clean.set_index(DATE_COL)[VARIABLE]

models = {
    "SARIMA":       ea.get_model("sarima",        order=(1,1,1), seasonal_order=(1,1,1,12)),
    "XGBoost":      ea.get_model("xgboost",       lags=[1,2,3,6,12]),
    "RandomForest": ea.get_model("random_forest", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    results[name] = ea.walk_forward(
        model, ts, horizon=HORIZONTE, n_splits=N_SPLITS, domain=DOMINIO
    )
    print(f"{name:15s} RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
# Ranking multi-criterio
ranking = ea.rank_models(results, domain=DOMINIO)
print(f"Mejor modelo: {ea.select_best(results, domain=DOMINIO)}")
ranking[["rmse","mae","r2","score","rank"]]

In [ ]:
ea.compare_backtests(results)

## 9. Reporte final

In [ ]:
# Pronóstico del mejor modelo en los últimos N periodos
N_TEST = HORIZONTE * 2
train, test = ts.iloc[:-N_TEST], ts.iloc[-N_TEST:]

best_name = ea.select_best(results, domain=DOMINIO)
best_model = models[best_name]
best_model.fit(train)
preds = {best_name: best_model.predict(N_TEST)}
mets  = {best_name: ea.evaluate(test.values, preds[best_name], domain=DOMINIO)}

rpt = ea.forecast_report(
    test, preds, mets,
    output=f"data/output/reports/forecast_{LINEA}.html",
    title=f"Pronóstico {VARIABLE} — {LINEA}",
    variable_name=VARIABLE, unit=UNIDAD,
)
print(f"Reporte: {rpt}")

In [ ]:
# Reporte estadístico descriptivo + inferencial
rpt_stats = ea.stats_report(
    df_clean,
    output=f"data/output/reports/stats_{LINEA}.html",
    title=f"Estadística — {LINEA}",
    date_col=DATE_COL,
)
print(f"Reporte estadístico: {rpt_stats}")

## 10. Registro de decisiones

Agregar al final de `docs/decisiones.md` cualquier decisión relevante:

```
## ADR-XXX — <Título>
**Fecha:** YYYY-MM-DD | **Estado:** Aceptado
**Contexto:** ...
**Decisión:** ...
**Consecuencias:** ...
```

Y actualizar `Plan.md` sección 11 (seguimiento de avance).